# Diyetisyen AI Bot - ChromaDB + Türkçe RAG

TÜBER 2022 (Türkiye Beslenme Rehberi) tabanlı akıllı beslenme asistanı.
   - TÜBER 2022 PDF'i rapordaki kaynakçadan bulunabilir.
   - Sistemin çalışması için proje içinde data_resources isimli bir directory içine bu pdf'i koymanız gerekir

### Özellikler
- **ChromaDB**
- **Türkçe optimize** multilingual embedding (E5)
- **Hybrid Search** (BM25 + Vector + RRF Fusion)
- **Semantic Chunking** (cümle bazlı)
- **PDF Metin Temizleme** (TextCleaner)
- **Qwen2-VL-2B** LLM ile yanıt üretimi

### Gerekli Dosyalar
- `config.yaml` - Ayar dosyası
- `data_resources/Turkiye_Beslenme_Rehber_TUBER_2022_min.pdf` - PDF dosyası

## 1. Kurulum
Gerekli kütüphaneleri kuruyoruz.

In [1]:
# gerekli kütüphaneler
!pip install -q transformers sentence-transformers rank_bm25 pdfplumber accelerate chromadb pyyaml

## 2.  Kütüphaneleri Yükle
Projede kullanılacak tüm Python kütüphanelerini import eder. Bunlar arasında dosya işleme (os, pathlib), metin işleme (re), veri yapıları (dataclasses), PDF okuma (pdfplumber), vektör veritabanı (chromadb), BM25 arama (rank_bm25), embedding modeli (sentence-transformers) ve LLM (transformers - Qwen2-VL) yer alır.


In [2]:
import os
import re
import sys
import logging
from pathlib import Path
from typing import List, Dict, Optional, Tuple, Any
from dataclasses import dataclass
from datetime import datetime

import yaml
import torch
import numpy as np
import pdfplumber
import chromadb
from chromadb.config import Settings
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

print("Kütüphaneler yüklendi")

C:\Users\USER\pytorch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Kütüphaneler yüklendi


## 3. Ayarlar ve GPU Kontrolü
Projenin tüm ayarlarını (dosya yolları, model isimleri, chunking parametreleri, LLM üretim parametreleri) CONFIG dictionary'sinde tanımlar. Ayrıca GPU kontrolü yaparak CUDA kullanılabilirliğini kontrol eder ve uygun cihazı (cuda/cpu) seçer.

In [3]:
# ============================================
# AYARLAR (config.yaml yerine inline)
# ============================================
CONFIG = {
    "paths": {
        "pdf": "data_resources/Turkiye_Beslenme_Rehber_TUBER_2022_min.pdf",
        "chroma_db": "data_resources/chroma_db"
    },
    "models": {
        "llm": "Qwen/Qwen2-VL-2B-Instruct",
        "embedding": "intfloat/multilingual-e5-base"
    },
    "chroma": {
        "collection_name": "tuber_2022"
    },
    "chunking": {
        "chunk_size": 300,
        "chunk_overlap": 50
    },
    "generation": {
        "max_new_tokens": 512,
        "temperature": 0.7,
        "top_p": 0.9,
        "repetition_penalty": 1.2,
        "no_repeat_ngram_size": 3
    },
    "retrieval": {
        "top_k": 3
    }
}

# GPU Kontrolü
def check_gpu() -> str:
    print(f"PyTorch: {torch.__version__}")
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print(f"GPU Aktif: {gpu_name}")
        return "cuda"
    else:
        print("GPU bulunamadı, CPU kullanılacak.")
        return "cpu"

DEVICE = check_gpu()

PyTorch: 2.6.0+cu124
GPU Aktif: NVIDIA GeForce RTX 4060


## 4. Data Classes
Projede kullanılacak veri yapılarını dataclass olarak tanımlar: PageContent (PDF sayfa içeriği), Chunk (metin parçası ve metadata), SearchResult (arama sonucu, skor ve kaynak bilgisi).


In [4]:
@dataclass
class PageContent:
    """Bir PDF sayfasının içeriğini temsil eder."""
    page_number: int
    text: str
    word_count: int

@dataclass
class Chunk:
    """Bir metin parçasını ve metadata'sını temsil eder."""
    id: str
    text: str
    metadata: Dict[str, Any]

@dataclass
class SearchResult:
    """Bir arama sonucunu temsil eder."""
    text: str
    metadata: Dict[str, Any]
    score: float
    source: str

print("Data classes tanımlandı")

Data classes tanımlandı


## 5. TextCleaner - PDF Metin Temizleme
PDF'den çıkarılan bozuk metinleri temizleyen TextCleaner sınıfını tanımlar. Boşluklu karakterleri düzeltir (T Ü R K İ Y E → TÜRKİYE), tablo kalıntılarını temizler, anlamsız satırları kaldırır ve gereksiz boşlukları normalleştirir.

In [5]:
class TextCleaner:
    """
    PDF'den çıkarılan metni temizler ve normalleştirir.

    Düzeltilen sorunlar:
    - Boşluklu karakterler (T Ü R K İ Y E -> TÜRKİYE)
    - Tablo/grafik kalıntıları
    - Gereksiz boşluklar
    - Anlamsız satırlar
    """

    KNOWN_SPACED_WORDS = {
        'T Ü R K İ Y E': 'TÜRKİYE',
        'B E S L E N M E': 'BESLENME',
        'R E H B E R İ': 'REHBERİ',
        'S U N U Ş': 'SUNUŞ',
        'Ö N S Ö Z': 'ÖNSÖZ',
        'İ Ç İ N D E K İ L E R': 'İÇİNDEKİLER',
        'B Ö L Ü M': 'BÖLÜM',
        'G İ R İ Ş': 'GİRİŞ',
        'S O N U Ç': 'SONUÇ',
        'K A Y N A K L A R': 'KAYNAKLAR',
        'E K L E R': 'EKLER',
        'T B S A': 'TBSA',
        'T Ü B E R': 'TÜBER',
    }

    @classmethod
    def clean(cls, text: str) -> str:
        if not text:
            return ""

        for spaced, fixed in cls.KNOWN_SPACED_WORDS.items():
            text = text.replace(spaced, fixed)

        text = cls._fix_spaced_characters(text)
        text = cls._remove_garbage(text)
        text = cls._remove_broken_word_sequences(text)
        text = cls._normalize_whitespace(text)
        text = cls._clean_broken_lines(text)

        return text.strip()

    @classmethod
    def clean_sentence(cls, sentence: str) -> str:
        if not sentence:
            return ""

        words = sentence.split()
        if len(words) < 3:
            return sentence

        total_chars = sum(len(w) for w in words)
        short_word_chars = sum(len(w) for w in words if len(w) <= 2 and w.isalpha())

        if total_chars > 0 and short_word_chars / total_chars > 0.5:
            return ""

        return sentence

    @classmethod
    def _fix_spaced_characters(cls, text: str) -> str:
        lines = text.split('\n')
        fixed_lines = []

        for line in lines:
            words = line.split()
            if len(words) > 3:
                single_chars = sum(1 for w in words if len(w) == 1 and w.isalpha())
                ratio = single_chars / len(words) if words else 0
                if ratio > 0.5:
                    fixed_lines.append(''.join(words))
                else:
                    fixed_lines.append(line)
            else:
                fixed_lines.append(line)

        return '\n'.join(fixed_lines)

    @classmethod
    def _remove_garbage(cls, text: str) -> str:
        text = re.sub(r'(\s*\.\s*){4,}', ' ', text)
        text = re.sub(r'[\|\+\-]{3,}', ' ', text)
        text = re.sub(r'(?:\s+\d{1,2}\s+){5,}', ' ', text)
        return text

    @classmethod
    def _remove_broken_word_sequences(cls, text: str) -> str:
        lines = text.split('\n')
        cleaned_lines = []

        for line in lines:
            line = line.strip()
            if not line:
                cleaned_lines.append(line)
                continue

            words = line.split()
            if len(words) < 3:
                cleaned_lines.append(line)
                continue

            avg_word_len = sum(len(w) for w in words) / len(words)
            short_words = sum(1 for w in words if len(w) <= 2)
            short_ratio = short_words / len(words)

            if avg_word_len < 2.5 and short_ratio > 0.5:
                continue

            single_letter_count = sum(1 for w in words if len(w) == 1 and w.isalpha())
            if single_letter_count >= 5 and single_letter_count / len(words) > 0.4:
                continue

            cleaned_lines.append(line)

        return '\n'.join(cleaned_lines)

    @classmethod
    def _normalize_whitespace(cls, text: str) -> str:
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        return '\n'.join(line.strip() for line in text.split('\n'))

    @classmethod
    def _clean_broken_lines(cls, text: str) -> str:
        lines = text.split('\n')
        cleaned_lines = []

        for line in lines:
            line = line.strip()
            if not line:
                if cleaned_lines and cleaned_lines[-1]:
                    cleaned_lines.append('')
                continue
            if len(line) < 3:
                continue
            if not any(c.isalnum() for c in line):
                continue
            alnum_count = sum(1 for c in line if c.isalnum())
            if len(line) > 10 and alnum_count / len(line) < 0.3:
                continue
            cleaned_lines.append(line)

        return '\n'.join(cleaned_lines)

print("TextCleaner hazır")

TextCleaner hazır


## 6. PDF Processor - Metin Çıkarma ve Chunking
PDF işleme sınıfını tanımlar. pdfplumber ile sayfa sayfa metin çıkarır, TextCleaner ile temizler, bölüm başlıklarını algılar ve semantic chunking uygular. Chunk'lar belirli kelime sayısına göre oluşturulur ve overlap ile birbirine bağlanır.


In [6]:
class PDFProcessor:
    """PDF'den metin çıkarma ve chunking işlemleri."""

    SENTENCE_ENDINGS = re.compile(r'(?<=[.!?])\s+')

    SECTION_PATTERNS = [
        re.compile(r'^(\d+\.)+\s+[A-ZÇĞİÖŞÜ]'),
        re.compile(r'^[A-ZÇĞİÖŞÜ]{2,}[A-ZÇĞİÖŞÜa-zçğıöşü\s]+$'),
        re.compile(r'^Bölüm\s+\d+', re.IGNORECASE),
    ]

    def __init__(self, pdf_path: str, chunk_size: int = 300, chunk_overlap: int = 50):
        self.pdf_path = Path(pdf_path)
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self._current_section = "Giriş"

    def extract_text_with_metadata(self) -> List[PageContent]:
        """PDF'den sayfa bazlı metin çıkarır ve temizler."""
        if not self.pdf_path.exists():
            raise FileNotFoundError(f"PDF bulunamadı: {self.pdf_path}")

        print(f"PDF işleniyor: {self.pdf_path.name}")
        pages = []

        with pdfplumber.open(self.pdf_path) as pdf:
            total = len(pdf.pages)
            for i, page in enumerate(pdf.pages, 1):
                raw_text = page.extract_text() or ""
                cleaned_text = TextCleaner.clean(raw_text)

                if cleaned_text.strip():
                    pages.append(PageContent(
                        page_number=i,
                        text=cleaned_text,
                        word_count=len(cleaned_text.split())
                    ))

                if i % 50 == 0 or i == total:
                    print(f"   İşlenen: {i}/{total} sayfa")

        print(f"{len(pages)} sayfa işlendi")
        return pages

    def _detect_section_title(self, text: str) -> Optional[str]:
        lines = text.strip().split('\n')
        for line in lines[:5]:
            line = line.strip()
            if len(line) < 100:
                for pattern in self.SECTION_PATTERNS:
                    if pattern.match(line):
                        return line
        return None

    def _split_into_sentences(self, text: str) -> List[str]:
        text = re.sub(r'\n+', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        sentences = self.SENTENCE_ENDINGS.split(text)

        clean_sentences = []
        for s in sentences:
            s = s.strip()
            if s:
                cleaned = TextCleaner.clean_sentence(s)
                if cleaned:
                    clean_sentences.append(cleaned)
        return clean_sentences

    def create_chunks(self, pages: List[PageContent]) -> List[Chunk]:
        """Sayfa içeriklerinden chunk'lar oluşturur."""
        print("Semantic chunking başlıyor...")

        chunks = []
        chunk_index = 0
        all_sentences = []

        for page in pages:
            detected = self._detect_section_title(page.text)
            if detected:
                self._current_section = detected

            sentences = self._split_into_sentences(page.text)
            for sentence in sentences:
                all_sentences.append((sentence, page.page_number, self._current_section))

        current_sentences, current_pages, current_sections = [], [], []
        current_word_count = 0

        for sentence, page_no, section in all_sentences:
            sentence_words = len(sentence.split())

            if current_word_count + sentence_words > self.chunk_size and current_sentences:
                chunk = self._create_chunk(
                    chunk_index, current_sentences,
                    current_pages, current_sections, current_word_count
                )
                chunks.append(chunk)
                chunk_index += 1

                # Overlap
                overlap_sentences, overlap_pages, overlap_sections = [], [], []
                overlap_words = 0

                for i in range(len(current_sentences) - 1, -1, -1):
                    s = current_sentences[i]
                    s_words = len(s.split())
                    if overlap_words + s_words <= self.chunk_overlap:
                        overlap_sentences.insert(0, s)
                        overlap_pages.insert(0, current_pages[i])
                        overlap_sections.insert(0, current_sections[i])
                        overlap_words += s_words
                    else:
                        break

                current_sentences = overlap_sentences
                current_pages = overlap_pages
                current_sections = overlap_sections
                current_word_count = overlap_words

            current_sentences.append(sentence)
            current_pages.append(page_no)
            current_sections.append(section)
            current_word_count += sentence_words

        if current_sentences:
            chunk = self._create_chunk(
                chunk_index, current_sentences,
                current_pages, current_sections, current_word_count
            )
            chunks.append(chunk)

        print(f"{len(chunks)} chunk oluşturuldu")
        return chunks

    def _create_chunk(self, idx, sentences, pages, sections, word_count) -> Chunk:
        section_counts = {}
        for s in sections:
            section_counts[s] = section_counts.get(s, 0) + 1
        primary_section = max(section_counts, key=section_counts.get)

        unique_pages = sorted(set(pages))
        page_range = f"{unique_pages[0]}-{unique_pages[-1]}" if len(unique_pages) > 1 else str(unique_pages[0])

        return Chunk(
            id=f"chunk_{idx:04d}",
            text=" ".join(sentences),
            metadata={
                "chunk_index": idx,
                "word_count": word_count,
                "sentence_count": len(sentences),
                "page_start": min(pages),
                "page_end": max(pages),
                "page_range": page_range,
                "section": primary_section,
                "created_at": datetime.now().isoformat()
            }
        )

    def process(self) -> List[Chunk]:
        pages = self.extract_text_with_metadata()
        return self.create_chunks(pages)

print("PDFProcessor hazır")

PDFProcessor hazır


## 7. ChromaDB Vector Store
ChromaDB vektör veritabanı sınıfını tanımlar. Multilingual E5 embedding modeli ile chunk'ları vektörlere dönüştürür, persistent olarak diske kaydeder ve cosine similarity ile semantic arama yapar.


In [7]:
class ChromaVectorStore:
    """ChromaDB ile persistent vector storage."""

    def __init__(self, db_path: str, collection_name: str, embedding_model: str):
        self.db_path = Path(db_path)
        self.collection_name = collection_name
        self.embedding_model_name = embedding_model

        print(f"Embedding: {embedding_model}")
        self.embedder = SentenceTransformer(embedding_model)
        print("Embedding modeli hazır")

        self.db_path.mkdir(parents=True, exist_ok=True)
        self.client = chromadb.PersistentClient(
            path=str(self.db_path),
            settings=Settings(anonymized_telemetry=False)
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        print(f"ChromaDB: {self.collection.count()} doküman")

    def add_documents(self, chunks: List[Chunk], force_reindex: bool = False) -> int:
        current_count = self.collection.count()

        if current_count > 0 and not force_reindex:
            print(f"Zaten indexlenmiş ({current_count} adet)")
            return 0

        if force_reindex and current_count > 0:
            print("Mevcut index siliniyor...")
            self.client.delete_collection(self.collection_name)
            self.collection = self.client.create_collection(
                name=self.collection_name,
                metadata={"hnsw:space": "cosine"}
            )

        print(f"{len(chunks)} chunk indexleniyor...")

        batch_size = 100
        for i in range(0, len(chunks), batch_size):
            batch = chunks[i:i + batch_size]
            ids = [c.id for c in batch]
            texts = [c.text for c in batch]
            metadatas = [c.metadata for c in batch]
            embeddings = self.embedder.encode(texts, show_progress_bar=False).tolist()
            self.collection.add(ids=ids, documents=texts, embeddings=embeddings, metadatas=metadatas)

        print(f"Indexleme tamamlandı: {self.collection.count()} doküman")
        return self.collection.count()

    def search(self, query: str, top_k: int = 3) -> List[SearchResult]:
        if "e5" in self.embedding_model_name.lower():
            query_text = f"query: {query}"
        else:
            query_text = query

        query_embedding = self.embedder.encode(query_text).tolist()

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )

        formatted = []
        for i in range(len(results["ids"][0])):
            formatted.append(SearchResult(
                text=results["documents"][0][i],
                metadata=results["metadatas"][0][i],
                score=1 - results["distances"][0][i],
                source="vector"
            ))
        return formatted

print("ChromaVectorStore hazır")

ChromaVectorStore hazır


## 8. 🔍 Hybrid Search Engine (BM25 + Vector + RRF)
Hybrid arama motorunu tanımlar. BM25 (kelime bazlı) ve Vector Search (semantic) sonuçlarını RRF (Reciprocal Rank Fusion) algoritması ile birleştirerek daha iyi arama sonuçları üretir.


In [8]:
class HybridSearchEngine:
    """BM25 + Vector + RRF Fusion."""

    def __init__(self, vector_store: ChromaVectorStore, chunks: List[Chunk]):
        self.vector_store = vector_store
        self.chunks = chunks
        self.chunk_texts = [c.text for c in chunks]

        print("BM25 indeksi oluşturuluyor...")
        tokenized = [doc.lower().split() for doc in self.chunk_texts]
        self.bm25 = BM25Okapi(tokenized)
        print("Hybrid search hazır")

    def _search_bm25(self, query: str, top_k: int) -> List[Tuple[int, float]]:
        scores = self.bm25.get_scores(query.lower().split())
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(int(idx), float(scores[idx])) for idx in top_indices]

    def _search_vector(self, query: str, top_k: int) -> List[Tuple[int, float]]:
        results = self.vector_store.search(query, top_k=top_k)
        return [(r.metadata.get("chunk_index", 0), r.score) for r in results]

    def _rrf(self, rankings: List[List[Tuple[int, float]]], k: int = 60) -> List[Tuple[int, float]]:
        rrf_scores = {}
        for ranking in rankings:
            for rank, (doc_idx, _) in enumerate(ranking):
                if doc_idx not in rrf_scores:
                    rrf_scores[doc_idx] = 0.0
                rrf_scores[doc_idx] += 1.0 / (k + rank + 1)
        return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    def search(self, query: str, top_k: int = 3, return_scores: bool = False):
        search_k = top_k * 2
        bm25_results = self._search_bm25(query, search_k)
        vector_results = self._search_vector(query, search_k)
        fused = self._rrf([bm25_results, vector_results])[:top_k]

        if return_scores:
            return [
                SearchResult(
                    text=self.chunk_texts[idx],
                    metadata=self.chunks[idx].metadata,
                    score=score,
                    source="hybrid"
                )
                for idx, score in fused
            ]
        return [self.chunk_texts[idx] for idx, _ in fused]

print("HybridSearchEngine hazır")

HybridSearchEngine hazır


## 9. LLM Answer Generator (Qwen2-VL)
Qwen2-VL LLM modelini yükleyen ve cevap üreten sınıfı tanımlar. Bulunan context'leri ve soruyu bir prompt template'ine yerleştirir, modelden Türkçe ve TÜBER bilgisine dayalı cevap üretir.


In [9]:
class AnswerGenerator:
    """Qwen2-VL ile cevap üretimi."""

    def __init__(self, model_name: str, device: str, gen_params: Dict):
        self.device = device
        self.gen_params = gen_params

        print(f"LLM: {model_name}")
        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map="auto"
        )
        self.processor = AutoProcessor.from_pretrained(model_name)
        print("LLM hazır")

    def generate(self, query: str, context: List[str], context_metadata: Optional[List[Dict]] = None) -> str:
        if context_metadata:
            parts = []
            for text, meta in zip(context, context_metadata):
                page_info = f"[Sayfa {meta.get('page_range', '?')}]"
                parts.append(f"{page_info}\n{text}")
            context_text = "\n\n---\n\n".join(parts)
        else:
            context_text = "\n\n---\n\n".join(context)

        system_prompt = """Sen uzman bir diyetisyen asistanısın. Türkiye Beslenme Rehberi (TÜBER) 2022 bilgilerini kullanarak soruları yanıtlıyorsun.

Kurallar:
1. Sadece verilen bağlam bilgisini kullan
2. Bağlamda bilgi yoksa belirt
3. Kısa ve öz cevap ver
4. Türkçe yanıt ver"""

        user_content = f"""Bağlam:\n{context_text}\n\nSoru: {query}\n\nCevap:"""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]

        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=[text], padding=True, return_tensors="pt").to(self.device)

        with torch.no_grad():
            generated_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.gen_params["max_new_tokens"],
                do_sample=True,
                temperature=self.gen_params["temperature"],
                top_p=self.gen_params["top_p"],
                repetition_penalty=self.gen_params["repetition_penalty"],
                no_repeat_ngram_size=self.gen_params["no_repeat_ngram_size"]
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        output = self.processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        return output[0].strip()

print("AnswerGenerator hazır")

AnswerGenerator hazır


## 10. Ana Chatbot Sınıfı
Tüm bileşenleri (PDFProcessor, ChromaVectorStore, HybridSearchEngine, AnswerGenerator) bir araya getiren ana chatbot sınıfını tanımlar. ask() metodu ile soru alır, hybrid search ile ilgili chunk'ları bulur ve LLM ile cevap üretir.


In [10]:
class DiyetisyenBot:
    """Ana chatbot sınıfı."""

    def __init__(self, config: Dict, force_reindex: bool = False):
        self.config = config
        self.device = DEVICE
        self.force_reindex = force_reindex

        self.chunks = []
        self.vector_store = None
        self.search_engine = None
        self.generator = None

        self._init_components()

    def _init_components(self):
        # 1. PDF işle
        pdf_processor = PDFProcessor(
            self.config["paths"]["pdf"],
            self.config["chunking"]["chunk_size"],
            self.config["chunking"]["chunk_overlap"]
        )
        self.chunks = pdf_processor.process()

        # 2. Vector store
        self.vector_store = ChromaVectorStore(
            self.config["paths"]["chroma_db"],
            self.config["chroma"]["collection_name"],
            self.config["models"]["embedding"]
        )
        self.vector_store.add_documents(self.chunks, force_reindex=self.force_reindex)

        # 3. Hybrid search
        self.search_engine = HybridSearchEngine(self.vector_store, self.chunks)

        # 4. LLM
        self.generator = AnswerGenerator(
            self.config["models"]["llm"],
            self.device,
            self.config["generation"]
        )

    def ask(self, query: str, return_sources: bool = False):
        results = self.search_engine.search(
            query,
            top_k=self.config["retrieval"]["top_k"],
            return_scores=True
        )

        context_texts = [r.text for r in results]
        context_metadata = [r.metadata for r in results]

        answer = self.generator.generate(query, context_texts, context_metadata)

        if return_sources:
            return answer, results
        return answer

print("DiyetisyenBot hazır")

DiyetisyenBot hazır


## 11. Botu Başlat!
DiyetisyenBot instance'ını oluşturarak sistemi başlatır. PDF işlenir, chunk'lar oluşturulur, ChromaDB'ye indexlenir, hybrid search hazırlanır ve LLM yüklenir. force_reindex=True ile mevcut index sıfırdan oluşturulur.


In [11]:
# İlk çalıştırmada force_reindex=True yapın
# Sonraki çalıştırmalarda False yapabilirsiniz

FORCE_REINDEX = True  # ChromaDB'yi sıfırdan oluştur

print("Diyetisyen Bot başlatılıyor...")
print("=" * 50)

bot = DiyetisyenBot(CONFIG, force_reindex=FORCE_REINDEX)

print("\n" + "=" * 50)
print("DİYETİSYEN BOTU HAZIR!")
print("=" * 50)

Diyetisyen Bot başlatılıyor...
PDF işleniyor: Turkiye_Beslenme_Rehber_TUBER_2022_min.pdf
   İşlenen: 50/430 sayfa
   İşlenen: 100/430 sayfa
   İşlenen: 150/430 sayfa
   İşlenen: 200/430 sayfa
   İşlenen: 250/430 sayfa
   İşlenen: 300/430 sayfa
   İşlenen: 350/430 sayfa
   İşlenen: 400/430 sayfa
   İşlenen: 430/430 sayfa
400 sayfa işlendi
Semantic chunking başlıyor...
608 chunk oluşturuldu
Embedding: intfloat/multilingual-e5-base
Embedding modeli hazır
ChromaDB: 0 doküman
608 chunk indexleniyor...


`torch_dtype` is deprecated! Use `dtype` instead!


Indexleme tamamlandı: 608 doküman
BM25 indeksi oluşturuluyor...
Hybrid search hazır
LLM: Qwen/Qwen2-VL-2B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.63s/it]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


LLM hazır

DİYETİSYEN BOTU HAZIR!


## 12. Soru Sor
Bota tek bir soru sorar ve cevabı alır. Cevapla birlikte kullanılan kaynakları (sayfa numaraları ve skorları) gösterir. SORU değişkenini değiştirerek farklı sorular sorulabilir.


In [12]:
# Sorunuzu buraya yazın
SORU = "Peynir faydalı mıdır?"

print(f"Soru: {SORU}")
print("Düşünüyor...")

answer, sources = bot.ask(SORU, return_sources=True)

print(f"\nCevap:\n{answer}")
print("\n" + "-" * 50)
print("\nKaynaklar:")
for i, src in enumerate(sources, 1):
    print(f"\n[{i}] Sayfa {src.metadata.get('page_range', '?')} (Skor: {src.score:.3f})")
    print(f"    {src.text[:150]}...")

Soru: Peynir faydalı mıdır?
Düşünüyor...

Cevap:
Peynir, genellikle kalitelli proteinerden oluşmaktadır ve bu yüzden sağlıklı bir besin kaynağı olarak kabul edilir. Ayrıca, bazı deneylerde peyniri tüketen kişilerin kan şekillendirilmiş olduğu görülmüştür. Ancıtkan, peynirimizin kalori sahibi olduğundan, özellikle yağlardan fazlasını içeren diğer besinlerinden farklı olarak, peyne bağlı olarak kaloriji artırmaya yardımcı olabilirdiği düşünülmekteydi. Ancunki zamanlarda, peyneteki kalori sayımı, peynesinin kalori salgılanmasından dolayı, peyenin kalori başına kaç kilogram gereksiniz mi? İşte peynirenin kaloritenizi hesaplamanız için aşağıdaki adımları izleyebilirsiniz:

1. Peynirin kalitesini kontrol edin.
   - Peyniri kesintisiz bir şekilde tüketmenize rağmen, peyntenin kalitsal değişkenliği hakkında bilgiye sahip olmak isterseniz, peýnetin kaliti hakkında bilgilendirmeden önce, peyonunun kalitesi hakkında bilgelendirmesini beklediğiniz duruma geçin.

2. Peypenin hangi türüdür?
   - Örn

## 13. İnteraktif Mod (Opsiyonel)
Sürekli soru-cevap döngüsü başlatır. Kullanıcı 'q' yazana kadar soru sorabilir, '/kaynak' komutu ile son cevabın kaynaklarını görebilir. KeyboardInterrupt ile de çıkılabilir.

In [13]:
print("\n" + "=" * 50)
print("İNTERAKTİF MOD")
print("Komutlar: q=çıkış, /kaynak=son kaynakları göster")
print("=" * 50 + "\n")

last_sources = []

while True:
    try:
        query = input("\nSoru: ").strip()

        if query.lower() in ["q", "exit", "çıkış", "quit"]:
            print("\nGörüşmek üzere!")
            break

        if not query:
            continue

        if query == "/kaynak":
            if last_sources:
                print("\nKaynaklar:")
                for i, src in enumerate(last_sources, 1):
                    print(f"\n[{i}] Sayfa {src.metadata.get('page_range', '?')} (Skor: {src.score:.3f})")
                    print(f"    {src.text[:150]}...")
            else:
                print("Henüz soru sorulmadı.")
            continue

        print("Düşünüyor...")
        answer, sources = bot.ask(query, return_sources=True)
        last_sources = sources

        print(f"\nCevap:\n{answer}")
        print("-" * 40)

    except KeyboardInterrupt:
        print("\n\nÇıkış yapılıyor...")
        break
    except Exception as e:
        print(f"Hata: {e}")


İNTERAKTİF MOD
Komutlar: q=çıkış, /kaynak=son kaynakları göster

Düşünüyor...

Cevap:
Yetişkin olan bir erkekte genellikle günlük enerjili yaklaşık 2,500 - 3,550 kalsiumlu kaloridir. İşte bu oranın 1 kişi başına günlük enerjisini nasıl hesaplamaniza yardımcı olacak şekilde:

Gündüz enerjinin %25'si: 2% = 2 x 1 kcal 
Sonuç : 2x1 + 2(2%)+ 2(x2%) + 3(3%) + ... = 11.5 kcal 

Bu oranın erken yaşımdaki erkekleri etkilemezse, genç erkek lerin günlük enerjinizi 1,520 -1,800 kalisiumlu kalsiyum kaloride çıkarırken, yaşlı erkeklere 1-1,250 -2,000 kalori getirirsiniz.
----------------------------------------

Görüşmek üzere!
